In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [31]:
# 1. Load Data
train_data = pd.read_csv('P1 Data/Consumer_Complaints_train.csv')
test_data = pd.read_csv('P1 Data/Consumer_Complaints_test_share.csv')

In [32]:
train_data.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2014-05-15,Credit card,NaN,Billing statement,NaN,NaN,NaN,Wells Fargo & Company,MI,48342,Older American,NaN,Web,2014-05-16,Closed with explanation,Yes,No,856103
1,2014-09-18,Bank account or service,(CD) Certificate of deposit,"Making/receiving payments, sending money",NaN,NaN,NaN,Santander Bank US,PA,18042,NaN,NaN,Referral,2014-09-24,Closed,Yes,No,1034666
2,2014-03-13,Credit reporting,NaN,Incorrect information on credit report,Account status,NaN,NaN,Equifax,CA,92427,NaN,NaN,Referral,2014-04-03,Closed with non-monetary relief,Yes,No,756363
3,2015-07-17,Credit card,NaN,Billing statement,NaN,"My credit card statement from US Bank, XXXX. X...",Company chooses not to provide a public response,U.S. Bancorp,GA,305XX,Older American,Consent provided,Web,2015-07-17,Closed with monetary relief,Yes,No,1474177
4,2014-11-20,Credit card,NaN,Transaction issue,NaN,NaN,NaN,Bank of America,MA,02127,NaN,NaN,Web,2014-11-28,Closed with explanation,Yes,No,1132572


In [33]:
test_data.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2014-01-18,Bank account or service,Cashing a check without an account,Deposits and withdrawals,NaN,NaN,NaN,Bank of America,CA,95691,NaN,NaN,Web,2014-01-17,Closed with explanation,Yes,675956
1,2016-03-31,Debt collection,Credit card,Cont'd attempts collect debt not owed,Debt was paid,NaN,NaN,"National Credit Adjusters, LLC",FL,32086,NaN,Consent not provided,Web,2016-03-31,Closed with explanation,Yes,1858795
2,2012-03-08,Mortgage,Conventional adjustable mortgage (ARM),"Loan servicing, payments, escrow account",NaN,NaN,NaN,Wells Fargo & Company,CA,94618,NaN,NaN,Web,2012-03-09,Closed without relief,Yes,32637
3,2016-01-07,Credit reporting,NaN,Unable to get credit report/credit score,Problem getting report or credit score,NaN,Company chooses not to provide a public response,"TransUnion Intermediate Holdings, Inc.",FL,33584,Older American,NaN,Postal mail,2016-01-12,Closed with non-monetary relief,Yes,1731374
4,2013-08-23,Mortgage,FHA mortgage,"Loan modification,collection,foreclosure",NaN,NaN,NaN,Bank of America,FL,33543,NaN,NaN,Web,2013-08-23,Closed with explanation,Yes,501487


In [34]:
train_data.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='object')

In [36]:
test_data.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Complaint ID'],
      dtype='object')

In [37]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478421 entries, 0 to 478420
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 478421 non-null  object
 1   Product                       478421 non-null  object
 2   Sub-product                   339948 non-null  object
 3   Issue                         478421 non-null  object
 4   Sub-issue                     185796 non-null  object
 5   Consumer complaint narrative  75094 non-null   object
 6   Company public response       90392 non-null   object
 7   Company                       478421 non-null  object
 8   State                         474582 non-null  object
 9   ZIP code                      474573 non-null  object
 10  Tags                          67206 non-null   object
 11  Consumer consent provided?    135487 non-null  object
 12  Submitted via                 478421 non-null  object
 13 

In [38]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119606 entries, 0 to 119605
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 119606 non-null  object
 1   Product                       119606 non-null  object
 2   Sub-product                   84854 non-null   object
 3   Issue                         119606 non-null  object
 4   Sub-issue                     46546 non-null   object
 5   Consumer complaint narrative  18557 non-null   object
 6   Company public response       22776 non-null   object
 7   Company                       119606 non-null  object
 8   State                         118681 non-null  object
 9   ZIP code                      118680 non-null  object
 10  Tags                          16871 non-null   object
 11  Consumer consent provided?    33864 non-null   object
 12  Submitted via                 119605 non-null  object
 13 

In [39]:
train_data.shape

(478421, 18)

In [40]:
test_data.shape

(119606, 17)

In [43]:
# 2. Data Preprocessing
# Handling missing data
imputer = SimpleImputer(strategy='most_frequent')

# Ensure column names are consistent before and after imputation
train_data_columns = train_data.columns
test_data_columns = test_data.columns

# Align columns in test_data with train_data
missing_cols = set(train_data_columns) - set(test_data_columns)
for col in missing_cols:
	test_data[col] = np.nan

# Reorder columns to match train_data
test_data = test_data[train_data_columns]

train_data = pd.DataFrame(imputer.fit_transform(train_data), columns=train_data_columns)
test_data = pd.DataFrame(imputer.transform(test_data), columns=train_data_columns)

In [44]:
# Date feature engineering
if 'Complaint Date' in train_data.columns:
    train_data['Complaint Date'] = pd.to_datetime(train_data['Complaint Date'])
    train_data['Complaint_Month'] = train_data['Complaint Date'].dt.month
    test_data['Complaint Date'] = pd.to_datetime(test_data['Complaint Date'])
    test_data['Complaint_Month'] = test_data['Complaint Date'].dt.month

In [45]:
# Text Data - TF-IDF Vectorization
if 'Consumer Complaint Narrative' in train_data.columns:
    tfidf = TfidfVectorizer(max_features=100)
    train_tfidf = tfidf.fit_transform(train_data['Consumer Complaint Narrative'].fillna(''))
    test_tfidf = tfidf.transform(test_data['Consumer Complaint Narrative'].fillna(''))

In [46]:
# Dropping non-informative columns
train_data.drop(['Consumer ID', 'ZIP Code', 'Consumer Complaint Narrative'], axis=1, inplace=True, errors='ignore')
test_data.drop(['Consumer ID', 'ZIP Code', 'Consumer Complaint Narrative'], axis=1, inplace=True, errors='ignore')

In [48]:
# Encoding categorical variables
label_encoders = {}
for col in train_data.select_dtypes(include='object').columns:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col].astype(str))
    label_encoders[col] = le

    # Handle unseen labels in test data
    test_data[col] = test_data[col].astype(str).apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

--------

**Step-1 : Import necessary libraries**

In [8]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

**Step-2 : Load the datasets**

**Step-3 : Basic Analysis of data**

In [10]:
train_df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2014-05-15,Credit card,NaN,Billing statement,NaN,NaN,NaN,Wells Fargo & Company,MI,48342,Older American,NaN,Web,2014-05-16,Closed with explanation,Yes,No,856103
1,2014-09-18,Bank account or service,(CD) Certificate of deposit,"Making/receiving payments, sending money",NaN,NaN,NaN,Santander Bank US,PA,18042,NaN,NaN,Referral,2014-09-24,Closed,Yes,No,1034666
2,2014-03-13,Credit reporting,NaN,Incorrect information on credit report,Account status,NaN,NaN,Equifax,CA,92427,NaN,NaN,Referral,2014-04-03,Closed with non-monetary relief,Yes,No,756363
3,2015-07-17,Credit card,NaN,Billing statement,NaN,"My credit card statement from US Bank, XXXX. X...",Company chooses not to provide a public response,U.S. Bancorp,GA,305XX,Older American,Consent provided,Web,2015-07-17,Closed with monetary relief,Yes,No,1474177
4,2014-11-20,Credit card,NaN,Transaction issue,NaN,NaN,NaN,Bank of America,MA,02127,NaN,NaN,Web,2014-11-28,Closed with explanation,Yes,No,1132572


In [11]:
test_df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2014-01-18,Bank account or service,Cashing a check without an account,Deposits and withdrawals,NaN,NaN,NaN,Bank of America,CA,95691,NaN,NaN,Web,2014-01-17,Closed with explanation,Yes,675956
1,2016-03-31,Debt collection,Credit card,Cont'd attempts collect debt not owed,Debt was paid,NaN,NaN,"National Credit Adjusters, LLC",FL,32086,NaN,Consent not provided,Web,2016-03-31,Closed with explanation,Yes,1858795
2,2012-03-08,Mortgage,Conventional adjustable mortgage (ARM),"Loan servicing, payments, escrow account",NaN,NaN,NaN,Wells Fargo & Company,CA,94618,NaN,NaN,Web,2012-03-09,Closed without relief,Yes,32637
3,2016-01-07,Credit reporting,NaN,Unable to get credit report/credit score,Problem getting report or credit score,NaN,Company chooses not to provide a public response,"TransUnion Intermediate Holdings, Inc.",FL,33584,Older American,NaN,Postal mail,2016-01-12,Closed with non-monetary relief,Yes,1731374
4,2013-08-23,Mortgage,FHA mortgage,"Loan modification,collection,foreclosure",NaN,NaN,NaN,Bank of America,FL,33543,NaN,NaN,Web,2013-08-23,Closed with explanation,Yes,501487


In [12]:
train_df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='object')

In [13]:
test_df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Complaint ID'],
      dtype='object')

In [14]:
train_df.shape

(478421, 18)

In [15]:
test_df.shape

(119606, 17)

In [16]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478421 entries, 0 to 478420
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 478421 non-null  object
 1   Product                       478421 non-null  object
 2   Sub-product                   339948 non-null  object
 3   Issue                         478421 non-null  object
 4   Sub-issue                     185796 non-null  object
 5   Consumer complaint narrative  75094 non-null   object
 6   Company public response       90392 non-null   object
 7   Company                       478421 non-null  object
 8   State                         474582 non-null  object
 9   ZIP code                      474573 non-null  object
 10  Tags                          67206 non-null   object
 11  Consumer consent provided?    135487 non-null  object
 12  Submitted via                 478421 non-null  object
 13 

In [17]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119606 entries, 0 to 119605
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 119606 non-null  object
 1   Product                       119606 non-null  object
 2   Sub-product                   84854 non-null   object
 3   Issue                         119606 non-null  object
 4   Sub-issue                     46546 non-null   object
 5   Consumer complaint narrative  18557 non-null   object
 6   Company public response       22776 non-null   object
 7   Company                       119606 non-null  object
 8   State                         118681 non-null  object
 9   ZIP code                      118680 non-null  object
 10  Tags                          16871 non-null   object
 11  Consumer consent provided?    33864 non-null   object
 12  Submitted via                 119605 non-null  object
 13 

**Step-4 : Feature Engineering**

* The below code describes the percentage of null values and as we can see some columns has null values above 50% , so we can drop these columns as these are not required. 

In [18]:
train_df.isnull().sum()/len(train_df)*100

Date received                    0.000000
Product                          0.000000
Sub-product                     28.943755
Issue                            0.000000
Sub-issue                       61.164748
Consumer complaint narrative    84.303783
Company public response         81.106181
Company                          0.000000
State                            0.802431
ZIP code                         0.804313
Tags                            85.952540
Consumer consent provided?      71.680382
Submitted via                    0.000000
Date sent to company             0.000000
Company response to consumer     0.000000
Timely response?                 0.000000
Consumer disputed?               0.000000
Complaint ID                     0.000000
dtype: float64

In [19]:
# Renaming columns in train_df
train_df.rename(columns={
    'Date received': 'Date_received',
    'Sub-product': 'Sub_product',
    'Sub-issue': 'Sub_issue',
    'Consumer complaint narrative': 'Consumer_complaint_narrative',
    'Company public response': 'Company_public_response',
    'ZIP code': 'ZIP_code',
    'Consumer consent provided?': 'Consumer_consent_provided',
    'Submitted via': 'Submitted_via',
    'Date sent to company': 'Date_sent_to_company',
    'Company response to consumer': 'Company_response_to_consumer',
    'Timely response?': 'Timely_response',
    'Consumer disputed?': 'Consumer_disputed'
}, inplace=True)

# Renaming columns in test_df
test_df.rename(columns={
    'Date received': 'Date_received',
    'Sub-product': 'Sub_product',
    'Sub-issue': 'Sub_issue',
    'Consumer complaint narrative': 'Consumer_complaint_narrative',
    'Company public response': 'Company_public_response',
    'ZIP code': 'ZIP_code',
    'Consumer consent provided?': 'Consumer_consent_provided',
    'Submitted via': 'Submitted_via',
    'Date sent to company': 'Date_sent_to_company',
    'Company response to consumer': 'Company_response_to_consumer',
    'Timely response?': 'Timely_response',
    'Complaint ID': 'Complaint_ID'
}, inplace=True)

In [20]:
train_df = train_df.drop(columns = ['Tags','Sub_issue','Company_public_response', 'Consumer_complaint_narrative','Consumer_consent_provided'], axis = 1)

In [21]:
train_df = train_df.drop(columns = ['Sub_product'], axis = 1)

In [22]:
train_df['State'].mode()
train_df['State'] = train_df['State'].fillna('CA')

In [23]:
train_df['ZIP_code'].mode()
train_df['ZIP_code'].fillna(train_df['ZIP_code'].mode())


0         48342
1         18042
2         92427
3         305XX
4         02127
          ...  
478416    77008
478417    36111
478418    79201
478419    91765
478420    77041
Name: ZIP_code, Length: 478421, dtype: object

**Step-5 : Correlationship between columns And Encoding of categorical columns**

In [24]:
df_num = train_df.select_dtypes(include = [np.number])
df_cat = train_df.select_dtypes(include = 'object')

In [25]:
df_num.columns

Index(['Complaint ID'], dtype='object')

In [26]:
df_cat.columns

Index(['Date_received', 'Product', 'Issue', 'Company', 'State', 'ZIP_code',
       'Submitted_via', 'Date_sent_to_company', 'Company_response_to_consumer',
       'Timely_response', 'Consumer_disputed'],
      dtype='object')

In [27]:
train_df['Consumer_disputed'].value_counts()

Consumer_disputed
No     376990
Yes    101431
Name: count, dtype: int64

In [28]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.preprocessing import LabelEncoder

# List of categorical columns
categorical_cols = ['Date_received', 'Product', 'Issue', 'Company', 'State', 'ZIP_code',
                    'Submitted_via', 'Date_sent_to_company', 'Company_response_to_consumer',
                    'Timely_response']

# Convert target column to binary format
train_df["Consumer_disputed"] = train_df["Consumer_disputed"].map({'Yes': 1, 'No': 0})

# Encode categorical columns
label_encoders = {}
for col in categorical_cols:
    train_df[col] = train_df[col].astype(str)  # Convert to string
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])  # Label encode
    label_encoders[col] = le

# Ensure 'Consumer_disputed?' has at least two categories
if train_df["Consumer_disputed"].nunique() < 2:
    print("ANOVA cannot be performed: Only one unique value in 'Consumer_disputed'")
else:
    # Perform One-Way ANOVA
    anova_results = {}

    for col in categorical_cols:
        groups = [train_df[train_df["Consumer_disputed"] == val][col].dropna() for val in train_df["Consumer_disputed"].unique()]

        # Ensure all groups have at least 2 values before performing ANOVA
        if len(groups) > 1 and all(len(g) > 1 for g in groups):
            f_stat, p_value = stats.f_oneway(*groups)
            anova_results[col] = p_value
        else:
            print(f"Skipping {col}: Not enough valid groups")

    # Convert results to DataFrame
    anova_df = pd.DataFrame(anova_results.items(), columns=['Feature', 'P-Value']).sort_values(by='P-Value')

    # Print significant features
    print(anova_df)


                        Feature       P-Value
6                 Submitted_via  0.000000e+00
8  Company_response_to_consumer  0.000000e+00
1                       Product  5.291092e-51
4                         State  6.255961e-12
5                      ZIP_code  1.058901e-07
0                 Date_received  1.075917e-02
9               Timely_response  2.066507e-02
7          Date_sent_to_company  3.070065e-02
3                       Company  6.639074e-01
2                         Issue  7.575411e-01
